# 05 — Prepare features, labels and modelling splits
 **Purpose:** predict whether next week's local median rises by at least **10%**, after current-week observations are available.
 **Saved:** three split Parquets, latest prediction inputs, settings JSON and small training comparison/summary CSVs. Detailed source audits are in optional Notebook 90.
 Features, preprocessing and model evaluation stay separate: train models in Notebook 06.


## 1. Load complete-scope weekly data and shared settings


In [ ]:
import pandas as pd
import numpy as np
import json
import pricecatcher_config as cfg

OUTPUT_DIR = cfg.STAGE5_DIR
TRAIN_END = pd.Timestamp(cfg.TRAIN_END_DATE)
VALIDATION_END = pd.Timestamp(cfg.VALIDATION_END_DATE)
TEST_END = pd.Period(cfg.END_MONTH, freq="M").end_time.normalize()
SERIES_KEYS = cfg.GROUP_KEYS
MIN_RECORDS, MIN_PREMISES = cfg.MIN_RECORDS, cfg.MIN_PREMISES
MIN_HISTORY_4W = cfg.MIN_HISTORY_4W
SURGE_THRESHOLD = cfg.SURGE_THRESHOLD_PCT
if not TRAIN_END < VALIDATION_END < TEST_END:
    raise ValueError("Date boundaries must be training < validation < test.")
if SURGE_THRESHOLD <= 0 or MIN_RECORDS < 1 or MIN_PREMISES < 1 or not 1 <= MIN_HISTORY_4W <= 4:
    raise ValueError("Check the surge and coverage settings.")
weekly = pd.read_parquet(cfg.STAGE4_WEEKLY_PARQUET)
weekly["week_start"] = pd.to_datetime(weekly["week_start"]).dt.normalize()
weekly["week_end"] = weekly["week_start"] + pd.Timedelta(days=6)
required = SERIES_KEYS + ["week_start", "weekly_median_price", "record_count",
                          "premise_count", "observed_days", "weekly_std_price", "weekly_min_price",
                          "weekly_max_price", "suspicious_record_count", "equal_premise_median"]
missing_columns = set(required) - set(weekly.columns)
if missing_columns:
    raise ValueError(f"Missing columns: {sorted(missing_columns)}")
if weekly.empty or weekly[SERIES_KEYS + ["week_start"]].isna().any().any():
    raise ValueError("Empty dataset or missing series/date identifiers.")
if weekly.duplicated(SERIES_KEYS + ["week_start"]).any():
    raise ValueError("Duplicate weekly series keys.")
if not weekly["week_start"].dt.dayofweek.eq(0).all():
    raise ValueError("week_start must be Monday.")
if not (np.isfinite(weekly["weekly_median_price"]) & weekly["weekly_median_price"].gt(0)).all():
    raise ValueError("Median prices must be finite and positive.")
scope_start = pd.Period(cfg.START_MONTH, freq="M").start_time.normalize()
if (weekly["week_start"] < scope_start).any() or (weekly["week_end"] > TEST_END).any():
    raise ValueError("Partial scope weeks found: use the complete_scope output from 04.")
if not (weekly["record_count"].ge(1) & weekly["premise_count"].ge(1) &
        weekly["observed_days"].between(1, 7)).all():
    raise ValueError("Invalid coverage counts.")
weekly = weekly.sort_values(SERIES_KEYS + ["week_start"]).copy()
print("Observed weekly rows:", len(weekly))
print("Local series:", weekly.groupby(SERIES_KEYS).ngroups)
print("Week range:", weekly["week_start"].min(), "to", weekly["week_start"].max())


## 2. Build exact-week features separately for each local series
 Insert missing calendar weeks temporarily. Never fill missing prices. Lags use 1, 2 and 4 calendar weeks; rolling features use past weeks only. The current price and current coverage are available inputs. Future prices and coverage are used only to establish historical labels.


In [ ]:
from pricecatcher_features import make_features

parts = []
for _, series in weekly.groupby(SERIES_KEYS, sort=False):
    parts.append(make_features(series))
featured = pd.concat(parts, ignore_index=True)
del parts
featured["week_end"] = featured["week_start"] + pd.Timedelta(days=6)
featured["next_week_start"] = featured["week_start"] + pd.Timedelta(days=7)
featured["next_week_end"] = featured["week_start"] + pd.Timedelta(days=13)
featured["month"] = featured["week_start"].dt.month
week_number = featured["week_start"].dt.isocalendar().week.astype(float)
featured["week_sin"] = np.sin(2 * np.pi * week_number / 52.1775)
featured["week_cos"] = np.cos(2 * np.pi * week_number / 52.1775)
print("Feature rows:", len(featured))


## 3. Define the outcome
A surge means an increase of at least the configured threshold (default 10%) in the observed local median between immediately consecutive Monday–Sunday weeks. It is a project definition, not an official economic standard. It can reflect premise composition and monitoring weights as well as within-premise prices.
Require two records and one premise in both weeks. Missing outcomes remain unlabelled. History requires a prior-week price and two observed prices in the preceding four calendar weeks. The global p95 is a training-only comparator, not the chosen target. Next-week coverage determines whether a historical outcome can be evaluated; it is never required to issue a prediction.


In [ ]:
featured["current_coverage_ok"] = (featured["record_count"] >= MIN_RECORDS) & (featured["premise_count"] >= MIN_PREMISES)
featured["target_observed"] = featured["next_week_price"].notna()
featured["target_coverage_ok"] = (featured["next_week_record_count"] >= MIN_RECORDS) & (featured["next_week_premise_count"] >= MIN_PREMISES)
featured["valid_price_pair"] = featured["current_coverage_ok"] & featured["target_observed"] & featured["target_coverage_ok"]
featured["history_ok"] = featured["lag_1w_price"].notna() & (featured["history_count_4w"] >= MIN_HISTORY_4W)
print("Rows without an observed next calendar week:", (~featured["target_observed"]).sum())
print("Rows with supported current/next-week prices:", featured["valid_price_pair"].sum())
reference_mask = featured["valid_price_pair"] & (featured["next_week_end"] <= TRAIN_END)
training_changes = featured.loc[reference_mask, "next_week_change_pct"]
if training_changes.empty:
    raise ValueError("No supported training outcomes. Inspect dates and coverage.")
training_p95 = float(training_changes.quantile(0.95, interpolation="linear"))
print("Training 95th percentile:", round(training_p95, 4), "%")
print("Common target threshold:", SURGE_THRESHOLD, "%")
featured["surge_label"] = pd.Series(pd.NA, index=featured.index, dtype="Int64")
valid = featured["valid_price_pair"]
featured.loc[valid, "surge_label"] = featured.loc[valid, "next_week_change_pct"].ge(SURGE_THRESHOLD).astype("Int64")
assert featured.loc[~valid, "surge_label"].isna().all()


## 4. Split chronologically with boundary protection
 Training outcomes end in 2025. Validation predictions occur after the training cutoff and outcomes end by March 2026. Test predictions occur after validation and outcomes end by June 2026. Exclude horizons crossing a cutoff. Earlier observed history can support later predictions.


In [ ]:
featured["split"] = "boundary_or_unlabelled"
usable = featured["surge_label"].notna() & featured["history_ok"]
train_mask = usable & (featured["next_week_end"] <= TRAIN_END)
validation_mask = usable & (featured["week_end"] > TRAIN_END) & (featured["next_week_end"] <= VALIDATION_END)
test_mask = usable & (featured["week_end"] > VALIDATION_END) & (featured["next_week_end"] <= TEST_END)
featured.loc[train_mask, "split"] = "train"
featured.loc[validation_mask, "split"] = "validation"
featured.loc[test_mask, "split"] = "test"
train = featured.loc[train_mask].copy()
validation = featured.loc[validation_mask].copy()
test = featured.loc[test_mask].copy()

split_rows = []
for name, frame in [("train", train), ("validation", validation), ("test", test)]:
    if frame.empty:
        raise ValueError(f"{name} split is empty. Inspect dates, history and coverage.")
    split_rows.append({"split": name, "rows": len(frame), "surges": int(frame["surge_label"].sum()),
                       "surge_rate_pct": float(frame["surge_label"].mean() * 100)})
assert train["next_week_end"].max() <= TRAIN_END
assert validation["week_end"].min() > TRAIN_END
assert validation["next_week_end"].max() <= VALIDATION_END
assert test["week_end"].min() > VALIDATION_END
assert test["next_week_end"].max() <= TEST_END
split_summary = pd.DataFrame(split_rows)
display(split_summary)


## 5. Declare model inputs
 Notebook 06 must use `experiment_settings.json` → `feature_columns` to build X. Fit encoding, imputation and scaling on training only. Item codes are categories. Missing historical features stay missing here.
 Weekly standard deviation is defined for eligible weeks with at least two records.


In [ ]:
CATEGORICAL_FEATURES = ["item_code", "state", "district", "premise_type", "project_group"]
NUMERIC_FEATURES = ["weekly_median_price", "lag_1w_price", "lag_2w_price", "lag_4w_price",
                    "weekly_price_change_pct", "rolling_mean_4w", "rolling_std_4w",
                    "rolling_mean_8w", "rolling_std_8w", "history_count_4w",
                    "weekly_std_price", "record_count",
                    "premise_count", "observed_days", "month", "week_sin", "week_cos"]
FEATURE_COLUMNS = CATEGORICAL_FEATURES + NUMERIC_FEATURES
assert not any(c.startswith("next_week") or c in ["surge_label", "split"] for c in FEATURE_COLUMNS)
if not np.isfinite(featured[NUMERIC_FEATURES].fillna(0).to_numpy(dtype=float)).all():
    raise ValueError("Non-finite feature values found.")
print("Features:", len(FEATURE_COLUMNS))
missing_features = train[FEATURE_COLUMNS].isna().sum()
display(missing_features.loc[missing_features > 0].rename("training_missing_count").to_frame())
export_columns = FEATURE_COLUMNS + ["item", "unit", "week_start", "next_week_end", "surge_label"]
assert len(export_columns) == len(set(export_columns))


## 6. Compare thresholds on training data
 Compare 5%, 10%, 15% and the training 95th percentile (with a 5% floor). Recompute after cleaning and use the exact value, not its rounded display. One CSV includes overall and group counts/rates for eligible pairs and model-ready pairs.
 Stable groups may have zero positives. Do not manufacture surge cases. Report group results alongside overall metrics.


In [ ]:
threshold_rows = []
training_reference = featured.loc[reference_mask]
# Recompute from this run; never reuse an old experiment's percentile.
p95_comparison = max(5.0, training_p95)
thresholds = [("5%", 5.0), ("10%", 10.0), ("15%", 15.0), ("training_p95", p95_comparison)]
for population, frame in [("eligible_pairs", training_reference), ("model_ready", train)]:
    groups = [("ALL", frame)] + list(frame.groupby("project_group"))
    for label, cutoff in thresholds:
        for group, part in groups:
            positives = int(part["next_week_change_pct"].ge(cutoff).sum())
            threshold_rows.append({"population": population, "project_group": group,
                "threshold_name": label, "threshold_pct": cutoff, "training_rows": len(part),
                "surges": positives, "surge_rate_pct": 100 * positives / len(part)})
threshold_comparison = pd.DataFrame(threshold_rows)
model_comparison = threshold_comparison.loc[threshold_comparison["population"].eq("model_ready")]
display(model_comparison.pivot(index="project_group", columns="threshold_name", values="surges")
        .reindex(columns=["5%", "10%", "15%", "training_p95"]))

training_groups = train.groupby("project_group")["surge_label"].agg(rows="size", surges="sum", surge_rate="mean")
training_groups["surge_rate_pct"] = training_groups.pop("surge_rate") * 100
display(training_groups)
for group, row in training_groups.iterrows():
    if row["surges"] == 0:
        print(group + ": no positive training cases; group-specific surge learning is unsupported.")
    elif row["surges"] < 20:
        print(group + ": very few positive training cases; interpret group results cautiously.")


## 7. Check aggregation and coverage on training data
Compare the main observed median with a median giving each premise equal weight, and compare the main coverage rule with at least two premises and three observed days in both weeks. These are diagnostics, not automatic changes to the target. Equal weighting does not fix changing shop composition. Select any policy before final evaluation; rerun downstream steps if it changes.

In [ ]:
comparison = featured.loc[reference_mask].copy()
next_coverage = weekly[SERIES_KEYS + ["week_start", "observed_days", "equal_premise_median"]].rename(
    columns={"week_start": "next_week_start", "observed_days": "next_observed_days",
             "equal_premise_median": "next_equal_premise_median"})
comparison = comparison.merge(next_coverage, on=SERIES_KEYS + ["next_week_start"],
                              how="left", validate="one_to_one")
comparison["equal_premise_change_pct"] = (
    comparison["next_equal_premise_median"] / comparison["equal_premise_median"] - 1) * 100
stronger = (comparison["premise_count"].ge(cfg.COMPARISON_MIN_PREMISES)
    & comparison["next_week_premise_count"].ge(cfg.COMPARISON_MIN_PREMISES)
    & comparison["observed_days"].ge(cfg.COMPARISON_MIN_DAYS)
    & comparison["next_observed_days"].ge(cfg.COMPARISON_MIN_DAYS))
comparison_rows = []
for population, mask in [("main_coverage", pd.Series(True, index=comparison.index)),
                          ("stronger_coverage", stronger)]:
    part = comparison.loc[mask]
    observed_labels = part["next_week_change_pct"].ge(SURGE_THRESHOLD)
    equal_labels = part["equal_premise_change_pct"].ge(SURGE_THRESHOLD)
    comparison_rows.append({"population": population, "training_pairs": len(part),
        "observed_median_surges": int(observed_labels.sum()),
        "equal_premise_surges": int(equal_labels.sum()),
        "labels_disagree": int(observed_labels.ne(equal_labels).sum()),
        "disagreement_pct": observed_labels.ne(equal_labels).mean() * 100,
        "retained_pairs_pct": 100 * len(part) / len(comparison)})
robustness = pd.DataFrame(comparison_rows)
display(robustness)
robustness.to_csv(OUTPUT_DIR / "training_aggregation_coverage_comparison.csv", index=False)


## 8. Save the modelling handoff
 Export only declared features, identifiers, dates and labels for modelling. Keep small comparison tables as report evidence. Future prices and future coverage never enter the feature list.


## 8a. Save latest eligible inputs for predictions
Use the last complete calendar week in the configured data scope, rather than silently predicting from stale local weeks. Require current coverage and historical support only. No future coverage or outcome is needed. This is a historical prototype snapshot unless the data period is updated.

In [ ]:
latest_week = TEST_END - pd.Timedelta(days=(TEST_END.dayofweek + 1) % 7 + 6)
prediction_mask = (featured["week_start"].eq(latest_week)
                   & featured["current_coverage_ok"] & featured["history_ok"])
prediction_columns = FEATURE_COLUMNS + ["item", "unit", "week_start", "week_end", "next_week_start", "next_week_end"]
latest_inputs = featured.loc[prediction_mask, prediction_columns].copy()
latest_inputs.to_parquet(OUTPUT_DIR / "latest_prediction_inputs.parquet", index=False)
print("Latest complete week:", latest_week.date(), "| Eligible prediction rows:", len(latest_inputs))
if latest_inputs.empty:
    print("No supported rows for this week. Inspect coverage; do not substitute stale weeks.")


In [ ]:
# Split exports contain only model features, identifiers/dates and the label.
for name, frame in [("train", train), ("validation", validation), ("test", test)]:
    frame[export_columns].to_parquet(OUTPUT_DIR / f"{name}.parquet", index=False)
threshold_comparison.to_csv(OUTPUT_DIR / "training_threshold_comparison.csv", index=False)
split_summary.to_csv(OUTPUT_DIR / "split_summary.csv", index=False)
training_groups.to_csv(OUTPUT_DIR / "training_group_summary.csv")
settings = {
    "target": "surge_label", "surge_threshold_pct": SURGE_THRESHOLD,
    "target_definition": "immediately following calendar-week observed median is at least threshold percent above current week",
    "target_status": cfg.TARGET_STATUS,
    "threshold_reason": "10% gives a common percentage meaning; training sensitivity is reported, not an economic optimum",
    "review_policy": "six suspected source records conservatively excluded; other flags retained",
    "prediction_time": "after the current week's observations are available",
    "training_p95_pct": training_p95,
    "training_reference_rows": len(training_changes),
    "train_end": cfg.TRAIN_END_DATE, "validation_end": cfg.VALIDATION_END_DATE,
    "test_end": str(TEST_END.date()),
    "minimum_records": MIN_RECORDS, "minimum_premises": MIN_PREMISES,
    "minimum_history_4w": MIN_HISTORY_4W,
    "remove_flagged_prices": cfg.REMOVE_FLAGGED_PRICES,
    "feature_columns": FEATURE_COLUMNS, "categorical_features": CATEGORICAL_FEATURES,
    "numeric_features": NUMERIC_FEATURES,
    "split_summary": split_summary.to_dict(orient="records"),
    "training_group_summary": training_groups.reset_index().to_dict(orient="records"),
    "aggregation": "median of all daily observations; not a fixed-premise price index",
    "coverage_comparison": "training-only; main eligibility unchanged",
}
(OUTPUT_DIR / "experiment_settings.json").write_text(json.dumps(settings, indent=2))
print("Saved modelling files to:", OUTPUT_DIR)


## Next: Notebook 06
Use this run's train/validation/test files and experiment_settings.json together. Select only feature_columns for X; fit encoding, imputation, scaling and imbalance handling on training only. Use validation for model settings and decision cutoffs; reserve test for final evaluation.
The target is the observed local median. Optional Notebook 90 examines composition and monitoring effects using matched premises. Retain these under this definition, without calling them confirmed within-premise surges. Same-premise changes are retained with cause unconfirmed. Six tenfold-pattern records are conservatively excluded, not corrected. The optional training sensitivity table quantifies that decision; remaining flagged records are retained as residual source uncertainty.
Oil/sugar have scarce or absent positives. Report group metrics and an always-no-surge baseline; keep Isolation Forest anomalies separate from supervised labels. No manually edited decision CSV or repeated top-20 approval is required. Keep diagnostics as evidence. Review new extreme patterns if they materially challenge these conclusions, rather than assuming every outlier is an error.
